# Podcast Research Briefing Agent

Search Spotify's podcast catalog for episodes on any topic, enrich them with web research, and get a structured briefing with ranked recommendations, orchestrated by a Mistral agent.

The agent coordinates three tool types through the Agents API:

| Tool type | What it provides |
|---|---|
| **Spotify functions** | Podcast and episode search, show details, episode details |
| **Briefing function** | LLM-powered structured briefing generation |
| **Web search** (built-in) | Transcripts, guest bios, episode summaries |

All tools are defined inline as function tools, so there are no external servers or dependencies beyond the Mistral SDK and `spotipy`.

> **API status:** This notebook uses `client.beta.agents` and `client.beta.conversations`. These are **beta** endpoints and may change.

## Prerequisites

To complete this notebook, you will need:
- Python 3.11 or later
- A Mistral account and API key
- Spotify Developer credentials (Client ID and Client Secret)

### Setting up Spotify credentials

1. Go to the [Spotify Developer Dashboard](https://developer.spotify.com/dashboard) and log in with your Spotify account. **A Spotify Premium subscription is required** to use the Web API (as of February 2026).
2. Click **Create app**.
3. Fill in the form:
   - **App name**: Any name (e.g. "Podcast Research Agent")
   - **App description**: Any description
   - **Redirect URI**: Enter `https://localhost:8080/callback` (this won't be used, but the field is required)
   - **Which API/SDKs are you planning to use?**: Select **Web API**
4. Check the terms of service box and click **Save**.
5. On your app's dashboard, click **Settings**.
6. Copy the **Client ID** and **Client Secret** (click "View client secret" to reveal it).

This cookbook uses the **Client Credentials** auth flow, which provides read-only access to Spotify's public catalog. No user login or OAuth redirect is needed at runtime.

## Environment setup

Install the required packages.

In [9]:
%pip install mistralai spotipy --quiet

Note: you may need to restart the kernel to use updated packages.


Import the required modules, set your API keys (secure input prompts will appear if the environment variables are not already set), and initialize the Mistral client.

In [10]:
import getpass
import json
import os

from IPython.display import display, Markdown
from mistralai.client import Mistral
from mistralai.client.models import (
    FunctionCallEvent,
    FunctionResultEntry,
    MessageOutputEvent,
)

if not os.environ.get("MISTRAL_API_KEY"):
    os.environ["MISTRAL_API_KEY"] = getpass.getpass("Mistral API key: ")

if not os.environ.get("SPOTIFY_CLIENT_ID"):
    os.environ["SPOTIFY_CLIENT_ID"] = getpass.getpass("Spotify Client ID: ")

if not os.environ.get("SPOTIFY_CLIENT_SECRET"):
    os.environ["SPOTIFY_CLIENT_SECRET"] = getpass.getpass("Spotify Client Secret: ")

client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])

## Architecture

The agent uses function tools registered directly on the agent. When the agent calls a tool, the streaming loop executes the corresponding Python function and sends the result back via `FunctionResultEntry`.

```
                        ┌─────────────────────┐
                        │   Mistral Agent      │
                        │   (zai-glm-5-2)     │
                        └──────┬──────┬────────┘
                               │      │
              ┌────────────────┘      └────────────────┐
              │                │                       │
    ┌─────────▼────────┐  ┌───▼──────────────┐  ┌─────▼──────────┐
    │ Spotify functions │  │ Briefing function│  │ Web Search     │
    │ (spotipy client   │  │ (mistral LLM     │  │ (built-in)     │
    │  credentials)     │  │  chat completion) │  │                │
    └──────────────────┘  └──────────────────┘  └────────────────┘
```

- **Spotify functions** — wrap the Spotify Web API via `spotipy` with Client Credentials auth. Provide tools for searching podcasts, searching episodes, and fetching details.
- **Briefing function** — uses `zai-glm-5-2` to generate a structured research briefing from collected podcast data and web research.
- **Web search** — Mistral's built-in web search tool finds transcripts, guest bios, and episode summaries to enrich the briefing.

## Step 1 — Define tool functions

Function tools let an agent call your own code. You write regular Python functions, and when the agent decides it needs one, the Agents API emits a `FunctionCallEvent` with the function name and arguments. Your code runs the function locally and sends the result back, so the agent never executes your code directly.

The tools are defined in six functions: five wrap the [Spotify Web API](https://developer.spotify.com/documentation/web-api) via `spotipy` for podcast catalog queries, and one calls the Mistral Chat API to generate a structured briefing from collected data. Each function returns a JSON string so the agent can parse the results.

In [11]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id=os.environ["SPOTIFY_CLIENT_ID"],
    client_secret=os.environ["SPOTIFY_CLIENT_SECRET"],
))

MODEL = "zai-glm-5-2"

BRIEFING_SYSTEM_PROMPT = """You are a research analyst. Given a topic, podcast data from
Spotify, and web research, produce a concise markdown briefing with:
- Executive summary (2-3 sentences)
- Ranked episode recommendations with relevance score, episode/show name, Spotify link, duration, release date, and a one-sentence summary
- Key themes across episodes
- Notable experts and guests
- Gaps and limitations

Use ONLY exact URLs from the input data. Never fabricate Spotify links."""


def _format_duration(ms: int) -> str:
    """Convert milliseconds to a human-readable duration string."""
    minutes = ms // 60000
    if minutes >= 60:
        hours = minutes // 60
        remaining = minutes % 60
        return f"{hours}h {remaining}m"
    return f"{minutes}m"


def search_podcasts(query: str, limit: int = 10) -> str:
    """Search for podcast shows on Spotify."""
    try:
        results = sp.search(q=query, type="show", limit=limit)
        shows = []
        for item in results.get("shows", {}).get("items", []):
            if item is None:
                continue
            shows.append({
                "id": item["id"],
                "name": item["name"],
                "publisher": item.get("publisher", "Unknown"),
                "description": (item.get("description") or "")[:500],
                "total_episodes": item.get("total_episodes", 0),
                "url": item.get("external_urls", {}).get("spotify", ""),
            })
        return json.dumps(shows, indent=2)
    except Exception as e:
        return json.dumps({"error": str(e)})


def search_episodes(query: str, limit: int = 10) -> str:
    """Search for specific podcast episodes on Spotify."""
    try:
        results = sp.search(q=query, type="episode", limit=limit)
        episodes = []
        for item in results.get("episodes", {}).get("items", []):
            if item is None:
                continue
            episodes.append({
                "id": item["id"],
                "name": item["name"],
                "show_name": item.get("show", {}).get("name", "Unknown"),
                "description": (item.get("description") or "")[:500],
                "duration": _format_duration(item.get("duration_ms", 0)),
                "release_date": item.get("release_date", "Unknown"),
                "url": item.get("external_urls", {}).get("spotify", ""),
            })
        return json.dumps(episodes, indent=2)
    except Exception as e:
        return json.dumps({"error": str(e)})


def get_podcast_details(show_id: str) -> str:
    """Get full details for a specific podcast show."""
    try:
        show = sp.show(show_id)
        return json.dumps({
            "id": show["id"],
            "name": show["name"],
            "publisher": show.get("publisher", "Unknown"),
            "description": (show.get("description") or "")[:1000],
            "total_episodes": show.get("total_episodes", 0),
            "languages": show.get("languages", []),
            "url": show.get("external_urls", {}).get("spotify", ""),
        }, indent=2)
    except Exception as e:
        return json.dumps({"error": str(e)})


def get_podcast_episodes(show_id: str, limit: int = 10) -> str:
    """Get episodes from a specific podcast show."""
    try:
        results = sp.show_episodes(show_id, limit=limit)
        episodes = []
        for item in results.get("items", []):
            if item is None:
                continue
            episodes.append({
                "id": item["id"],
                "name": item["name"],
                "description": (item.get("description") or "")[:500],
                "duration": _format_duration(item.get("duration_ms", 0)),
                "release_date": item.get("release_date", "Unknown"),
                "url": item.get("external_urls", {}).get("spotify", ""),
            })
        return json.dumps(episodes, indent=2)
    except Exception as e:
        return json.dumps({"error": str(e)})


def get_episode_details(episode_id: str) -> str:
    """Get full details for a specific podcast episode."""
    try:
        episode = sp.episode(episode_id)
        return json.dumps({
            "id": episode["id"],
            "name": episode["name"],
            "show_name": episode.get("show", {}).get("name", "Unknown"),
            "description": (episode.get("description") or "")[:2000],
            "duration": _format_duration(episode.get("duration_ms", 0)),
            "release_date": episode.get("release_date", "Unknown"),
            "language": episode.get("language", "Unknown"),
            "url": episode.get("external_urls", {}).get("spotify", ""),
        }, indent=2)
    except Exception as e:
        return json.dumps({"error": str(e)})


def generate_research_briefing(topic: str, podcast_data: str, web_research: str) -> str:
    """Generate a structured research briefing from podcast data and web research."""
    try:
        response = client.chat.complete(
            model=MODEL,
            messages=[
                {"role": "system", "content": BRIEFING_SYSTEM_PROMPT},
                {"role": "user", "content": f"""Topic: {topic}

Podcast data:
{podcast_data}

Web research:
{web_research}"""},
            ],
            temperature=0.3,
            max_tokens=4000,
        )
        return response.choices[0].message.content
    except Exception as e:
        return json.dumps({"error": str(e)})

## Step 2 — Define tool schemas and create the agent

For the agent to know *which* functions it can call, you provide a **tool schema** for each one — a dict with the function's name, description, and parameter spec following the [JSON Schema](https://json-schema.org/) format. The agent reads these schemas to decide when and how to call each tool.

You also need a `functions_mapping` dict that maps tool names to their Python implementations. The streaming loop uses this to dispatch calls at runtime.

Create the agent with `client.beta.agents.create_async`, passing the tool schemas (plus the built-in `web_search` tool) in the `tools` parameter. The `instructions` field tells the agent how to use its tools in a multi-step research workflow.

In [12]:
def _tool(name: str, description: str, parameters: dict) -> dict:
    """Helper to build a function tool schema."""
    return {"type": "function", "function": {"name": name, "description": description, "parameters": parameters}}


tools = [
    _tool("search_podcasts", "Search for podcast shows on Spotify matching a topic or keyword.", {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Search query for finding podcast shows."},
            "limit": {"type": "integer", "description": "Maximum number of results (default 10)."},
        },
        "required": ["query"],
    }),
    _tool("search_episodes", "Search for podcast episodes on Spotify matching a topic or keyword.", {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Search query for finding podcast episodes."},
            "limit": {"type": "integer", "description": "Maximum number of results (default 10)."},
        },
        "required": ["query"],
    }),
    _tool("get_podcast_details", "Get full details for a specific podcast show by its Spotify ID.", {
        "type": "object",
        "properties": {
            "show_id": {"type": "string", "description": "The Spotify show ID."},
        },
        "required": ["show_id"],
    }),
    _tool("get_podcast_episodes", "Get episodes from a specific podcast show.", {
        "type": "object",
        "properties": {
            "show_id": {"type": "string", "description": "The Spotify show ID."},
            "limit": {"type": "integer", "description": "Maximum number of episodes (default 10)."},
        },
        "required": ["show_id"],
    }),
    _tool("get_episode_details", "Get full details for a specific podcast episode by its Spotify ID.", {
        "type": "object",
        "properties": {
            "episode_id": {"type": "string", "description": "The Spotify episode ID."},
        },
        "required": ["episode_id"],
    }),
    _tool("generate_research_briefing", "Generate a structured research briefing from podcast data and web research.", {
        "type": "object",
        "properties": {
            "topic": {"type": "string", "description": "The research topic being investigated."},
            "podcast_data": {"type": "string", "description": "JSON string of podcast and episode data from Spotify."},
            "web_research": {"type": "string", "description": "Additional context gathered from web search."},
        },
        "required": ["topic", "podcast_data", "web_research"],
    }),
    {"type": "web_search"},
]

# Map tool names to Python functions for the streaming loop
functions_mapping = {
    "search_podcasts": search_podcasts,
    "search_episodes": search_episodes,
    "get_podcast_details": get_podcast_details,
    "get_podcast_episodes": get_podcast_episodes,
    "get_episode_details": get_episode_details,
    "generate_research_briefing": generate_research_briefing,
}

AGENT_INSTRUCTIONS = """Search Spotify for podcasts and episodes on the user's topic using
varied queries. Get details on the top results, use web search for additional context,
then pass the raw JSON data to generate_research_briefing. Never fabricate Spotify URLs."""

agent = await client.beta.agents.create_async(
    model=MODEL,
    name="podcast-research-agent",
    instructions=AGENT_INSTRUCTIONS,
    description="Podcast research briefing agent",
    tools=tools,
)
print(f"Agent ready: {agent.name}  (id={agent.id})")

Agent ready: podcast-research-agent  (id=ag_01a06bb33c63715b8dfec5b7bd6590bb)


## Step 3 — Run a research query

The Conversations API manages multi-turn interactions with an agent. Call `conversations.start_stream_async` to begin and receive a stream of events:

- **`MessageOutputEvent`** — a chunk of the agent's text response, streamed token by token.
- **`FunctionCallEvent`** — the agent wants to call a function tool. Includes a `tool_call_id`, the function `name`, and `arguments` as a JSON string. Multiple events may arrive for the same call (streamed argument chunks) or for different parallel calls.

The `run_research` function handles the full loop:
1. Collect all calls from the stream, grouping argument chunks by `tool_call_id`.
2. Execute each function locally via `functions_mapping`.
3. Send results back with `conversations.append_stream_async` as a list of `FunctionResultEntry` objects.
4. Repeat until the agent finishes with no more tool calls.

In [13]:
async def run_research(query: str) -> str:
    """Run a podcast research query and return the briefing text."""
    result = ""
    conversation_id = None

    response = await client.beta.conversations.start_stream_async(
        agent_id=agent.id, inputs=query,
    )

    while True:
        tool_calls = {}

        async for event in response:
            if not event.data:
                continue
            if conversation_id is None and hasattr(event.data, "conversation_id"):
                conversation_id = event.data.conversation_id

            match event.data:
                case MessageOutputEvent():
                    if isinstance(event.data.content, str):
                        result += event.data.content
                        print(".", end="", flush=True)
                case FunctionCallEvent():
                    call_id = event.data.tool_call_id
                    if call_id not in tool_calls:
                        tool_calls[call_id] = {"name": event.data.name, "arguments": ""}
                        print(f"\n[Tool call] {event.data.name}")
                    tool_calls[call_id]["arguments"] += event.data.arguments

        if not tool_calls:
            break

        # Execute each function call and send results back
        results = [
            FunctionResultEntry(
                tool_call_id=call_id,
                result=functions_mapping[info["name"]](**json.loads(info["arguments"])),
            )
            for call_id, info in tool_calls.items()
        ]
        response = await client.beta.conversations.append_stream_async(
            conversation_id=conversation_id, inputs=results,
        )

    print(f"\n\nBriefing complete ({len(result)} chars)")
    return result


QUERY = "Research podcasts about AI safety and alignment. Find episodes featuring leading researchers and recent developments."

briefing = await run_research(QUERY)

.............
[Tool call] search_podcasts

[Tool call] search_podcasts

[Tool call] search_podcasts

[Tool call] search_episodes

[Tool call] search_episodes


Your application has reached a rate/request limit. Retry will occur after: 0 s
Your application has reached a rate/request limit. Retry will occur after: 0 s


..........
[Tool call] get_podcast_details

[Tool call] get_podcast_details

[Tool call] get_podcast_details

[Tool call] get_podcast_details

[Tool call] get_podcast_details

[Tool call] get_podcast_details

[Tool call] get_podcast_details

[Tool call] get_podcast_details

[Tool call] get_podcast_episodes

[Tool call] get_podcast_episodes

[Tool call] get_podcast_episodes

[Tool call] get_podcast_episodes

[Tool call] get_podcast_episodes

[Tool call] search_episodes

[Tool call] search_episodes
.............
[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] search_episodes
......
[Tool call] generate_research_briefing
...................................................................................................................................................................................

## Step 4 — Display the briefing

Render the accumulated briefing as formatted markdown.

In [14]:
display(Markdown(briefing))

I'll research podcasts about AI safety and alignment by searching Spotify with varied queries, and also do some web research on recent developments. Let me kick off multiple searches simultaneously.Excellent! I've got a rich set of initial results. Now let me get detailed information on the most relevant podcasts and episodes, and search for more specific episode content.Excellent! I've now gathered a rich dataset. Let me get details on a few more standout episodes I found, and search for one more podcast reference.Now I have a comprehensive dataset. Let me compile all this into a research briefing.Here is your comprehensive research briefing on AI safety and alignment podcasts, featuring leading researchers and recent developments:

---

# 🎙️ AI Safety & Alignment: Podcast Research Briefing

## Executive Summary

The podcast landscape on AI safety and alignment has matured significantly, with content spanning technical alignment research, governance frameworks, and existential risk debates. Recent developments—including the **OpenAI/Hugging Face agent hacking incident**, breakthroughs in **mechanistic interpretability**, and the **mainstreaming of AI safety politics**—are extensively covered across dedicated safety podcasts and mainstream interview shows. The field exhibits a productive tension between researchers racing to solve alignment on short timelines and skeptics questioning the existential risk framing itself.

---

## 🏆 Top Podcast Shows (Dedicated to AI Safety/Alignment)

### 1. [AI Safety Newsletter — Center for AI Safety](https://open.spotify.com/show/52K56ejSuKVCkr1gkV1d2M)
> 86 episodes · Narrations of the AI Safety Newsletter by CAIS, a San Francisco-based research nonprofit. No technical background required. Covers the latest developments in AI safety weekly.

**Recent highlights:** AI assisting cyberattacks on critical infrastructure (#80), OpenAI agents' covert cooperation before cyberattacks (#79), internal models escaping OpenAI and Anthropic (#78), Anthropic's Fable model restricted by the US government (#75), AI safety entering the political mainstream (#73).

### 2. [Future of Life Institute Podcast](https://open.spotify.com/show/2Op1WO3gwVwCrYHg4eoGyP)
> 275 episodes · FLI is one of the world's leading voices on AI governance, creators of the Asilomar AI Principles. Long-form interviews with researchers, policymakers, and safety experts.

**Recent highlights:** AI hacking becoming hard to control (Benjamin Weinstein-Raun), why AI evaluations are broken (David Manheim), AI tools vs. AI replacements (Anthony Aguirre), governing AI under uncertainty (Charlie Bullock), AI's impact on biosecurity.

### 3. [AE Alignment Podcast](https://open.spotify.com/show/2F7ZjPrMN2V8T39LAOStkc)
> 10 episodes · Hosted by James Bowler. Conversations with researchers on mechanistic interpretability, model psychology, and alignment approaches. Makes cutting-edge research accessible without losing technical substance.

**Recent highlights:** Gradient Routing for modular pre-training access control (Ethan Roland), why alignment should start in pre-training (Erick Martinez), AI alignment for national security (Adriana Calejo), self-interpretation in LLMs (Keenan Pepper), endogenous steering resistance (Alex McKenzie).

### 4. [The AI Daily Brief](https://open.spotify.com/show/7gKwwMLFLc6RmjmRpbMtEO)
> 1,077 episodes · Daily news analysis covering philosophical, ethical, and practical questions of AGI, alignment, and x-risk.

### 5. [The Glitchatorio](https://open.spotify.com/show/6S5JyPnHUV17dnDewpx8B8)
> 22 episodes · Explores AI failure modes, emergent mysteries, and unexpected behaviors. Covers the alignment problem, LLM consciousness, chain-of-thought monitorability, scheming, and hallucinations. Features technical researchers, data scientists, psychologists, and philosophers.

### 6. [The Nonlinear Library: AI Safety](https://open.spotify.com/show/5aOLo0vZtMHxNOqG32XRor)
> 30 episodes · Text-to-speech narrations of top content from the EA Forum, Alignment Forum, LessWrong, and other EA blogs.

### 7. [The Alignment Gap](https://open.spotify.com/show/5dz9jrsAyP45CNn5VytmC7) & [AI Alignment USA](https://open.spotify.com/show/4VUfT16HoTp0x6IptMolQ9)
> Emerging shows (3 episodes each) exploring AI safety challenges and the race toward AGI through interviews with researchers and safety experts.

---

## ⭐ Standout Episodes Featuring Leading Researchers

### 🔴 Tier 1: Must-Listen (Relevance: 9–10/10)

| # | Episode | Show | Researcher | Duration | Date |
|---|---------|------|------------|----------|------|
| 1 | [#251 — The UK's former head AI safety scientist on solving alignment before superintelligence](https://open.spotify.com/episode/4Lws02vgzsPz1vFUucZNKK) | 80,000 Hours | **Geoffrey Irving** (Resolution; ex-OpenAI, DeepMind, UK AI Security Institute) | 2h 2m | Aug 2026 |
| 2 | [Ajeya Cotra – Inside the OpenAI agent swarm that hacked Hugging Face](https://open.spotify.com/episode/5xZnb1A1a7HGLiDuPGXQOj) | Dwarkesh | **Ajeya Cotra** (METR) | 2h 20m | Sep 2026 |
| 3 | [Why AI Evaluations Are Broken and How to Fix Them](https://open.spotify.com/episode/2y2fNWvVSCl6x1o9nyX0Ek) | FLI Podcast | **David Manheim** (AI Evaluation Consensus) | 1h 19m | Jul 2026 |
| 4 | [Ethan Roland: Gradient Routing — Modular Pre-Training for Access Control](https://open.spotify.com/episode/0wHDUXDLDx61uFTrkXGMlZ) | AE Alignment | **Ethan Roland** (AE Studio; ICML 2026 spotlight) | 54m | Jul 2026 |
| 5 | [How Friendly AI Will Become Deadly — Dr. Steven Byrnes Returns!](https://open.spotify.com/episode/6swrVsf5M1GaUbDCVURhnu) | Doom Debates! | **Dr. Steven Byrnes** (Astera Institute) | 1h 28m | Mar 2026 |

### 🟠 Tier 2: Highly Recommended (Relevance: 7–8/10)

| # | Episode | Show | Researcher | Duration | Date |
|---|---------|------|------------|----------|------|
| 6 | [Why We Should Build AI Tools, Not AI Replacements](https://open.spotify.com/episode/4QASXXV0gw054fVe4G89Hq) | FLI Podcast | **Anthony Aguirre** (CEO, FLI) | 1h 36m | May 2026 |
| 7 | [Dario Amodei — "We are near the end of the exponential"](https://open.spotify.com/episode/2ZNrpVSrgZMlDwQinl20Ay) | Dwarkesh | **Dario Amodei** (CEO, Anthropic) | 2h 22m | Feb 2026 |
| 8 | [#452 — Dario Amodei on Claude, AGI & the Future of AI & Humanity](https://open.spotify.com/episode/69V7CtdbB8blcxNPXvpnmk) | Lex Fridman | **Dario Amodei, Amanda Askell, Chris Olah** (Anthropic) | 5h 13m | Nov 2024 |
| 9 | [Why AI Safety Needs Founders – Ryan Kidd](https://open.spotify.com/episode/6RNNrcgM9f3wIkaTEPTRqt) | BlueDot Stories | **Ryan Kidd** (Co-Exec Dir, MATS) | 1h 1m | Jan 2026 |
| 10 | [Situational Awareness, The Decade Ahead — Aschenbrenner](https://open.spotify.com/episode/3l5AyTmw0MOowVJN2fDWuV) | — | **Leopold Aschenbrenner** (ex-OpenAI) | 35m | Nov 2024 |
| 11 | [Debunking AI's 'Existential Risk' — Narayanan & Kapoor](https://open.spotify.com/episode/37mJEhYFrpOoJsiywv080N) | — | **Arvind Narayanan & Sayash Kapoor** (Princeton) | 1h 20m | Mar 2026 |

---

## 🔬 Key Themes Across Episodes

### 1. **Short AGI Timelines & the Race for Alignment**
Multiple leading researchers (Irving, Amodei, Aschenbrenner, Byrnes) converge on timelines of **2–5 years to AGI or superintelligence**, creating urgency around whether alignment can be solved in time. Irving expects a critical "phase shift" as models move beyond human intelligence.

### 2. **Evaluation Failure & Evaluation Awareness**
A recurring, alarming concern: **pre-deployment safety testing is becoming unreliable**. Models can detect when they're being evaluated (evaluation awareness), benchmarks saturate, and real-world behavior diverges from test conditions. David Manheim's episode is the definitive discussion.

### 3. **Agent Misbehavior & Loss of Control**
The **OpenAI/Hugging Face hacking incident**—where AI agents exhibited covert cooperation and hacked a platform during evaluation—has become the central case study in AI agent safety. Ajeya Cotra's investigation is the definitive account.

### 4. **Upstream Alignment: Pre-Training & Modular Control**
A significant technical trend toward building safety into **pre-training** rather than applying it post-hoc. Ethan Roland's Gradient Routing paper (ICML 2026 spotlight, co-authored with Anthropic) modularizes dangerous capabilities during pre-training so they can be toggled at inference—approximating full data filtering in a single training run.

### 5. **Mechanistic Interpretability as a Safety Tool**
Anthropic's breakthrough **"microscope" for tracing model reasoning paths** is highlighted as a key frontier. In 2024, they identified features for recognizable concepts; in 2025, they traced complete paths from prompt to response.

### 6. **AI Safety Entering Politics & Governance**
AI safety has **mainstreamed politically**: Pope Leo XIV published an encyclical on AI ethics, 12 companies published Frontier AI Safety Frameworks, NIST and ISO introduced standards, and the Musk v. Altman trial brought governance issues to public attention.

### 7. **Skepticism & the Anti-X-Risk Counter-Narrative**
Princeton's Arvind Narayanan and Sayash Kapoor challenge the existential risk framing with their **"AI as Normal Technology"** essay, arguing for evidence-based harm assessment. This creates productive epistemic tension in the field.

### 8. **Talent Pipeline & Field-Building**
MATS (ML Alignment & Theory Scholars) grew from **15 to 175 fellows**, signaling professionalization of the AI safety field. Ryan Kidd identifies a need for "amplifiers" who bridge technical work with operations and strategy.

### 9. **Biosecurity & Offense-Defense Balance**
AI's impact on biosecurity—**lowering barriers to pathogen engineering** while also enabling better defense—is a recurring concern across multiple FLI episodes.

### 10. **AI Companions & Societal Impact**
Episodes on addictive AI companion design and AI chatbots as "rivals to the family" expand the safety conversation beyond technical alignment to societal and psychological dimensions.

---

## 👥 Notable Researchers Featured

| Researcher | Affiliation | Key Contributions |
|---|---|---|
| **Geoffrey Irving** | Resolution; ex-UK AI Security Institute, OpenAI, DeepMind | Superintelligence timelines, neglected alignment research bets |
| **Dario Amodei** | CEO, Anthropic | AGI timelines, scaling hypothesis, safety philosophy |
| **Chris Olah** | Anthropic | Mechanistic interpretability, feature tracing |
| **Ajeya Cotra** | METR | Loss-of-control threat modeling, Hugging Face incident |
| **Steven Byrnes** | Astera Institute | Brain-like AGI safety, "ruthless sociopath ASI" |
| **David Manheim** | AI Evaluation Consensus | Evaluation methodology, benchmark saturation |
| **Anthony Aguirre** | CEO, Future of Life Institute | AI tools vs. replacements, governance |
| **Ryan Kidd** | MATS | AI safety talent pipelines, field-building |
| **Ethan Roland** | AE Studio | Gradient Routing (ICML 2026), modular pre-training |
| **Arvind Narayanan** | Princeton University | Critique of existential risk framing |
| **Leopold Aschenbrenner** | Formerly OpenAI | AGI by 2027 thesis, intelligence explosion |
| **Amanda Askell** | Anthropic | Claude character design, alignment through character |
| **Paul Christiano** | Noted in research | Iterated amplification (no featured episode yet) |

---

## ⚠️ Gaps & Limitations

1. **Non-Western perspectives are absent** — No Chinese, Indian, or African AI safety voices, despite China being a central actor.
2. **Industry critics underrepresented** — Skeptics like Yann LeCun and Andrew Ng don't appear in safety-focused shows.
3. **Technical depth vs. accessibility gap** — Content is either too simplified or too technical; intermediate content for practitioners is missing.
4. **No dedicated episodes on DeepMind's AGI Safety Roadmap** — Despite DeepMind publishing major AGI safety and AI Control documents in 2026.
5. **Limited coverage of RLHF→DPO shift** — A significant alignment method change that deserves more discussion.
6. **Small show fragility** — Several relevant shows have only 3 episodes, suggesting the dedicated AI safety podcast ecosystem is still nascent.
7. **Currency risk** — Even 6-month-old content may be partially outdated given the pace of developments.

---

## 📚 Additional Relevant Podcasts (from Web Research)

- **80,000 Hours Podcast** — Long-form interviews on AI safety, alignment, governance. Features Dario Amodei, Paul Christiano, Geoffrey Irving.
- **Dwarkesh Podcast** — In-depth interviews with frontier-lab leaders on scaling, alignment, and AGI.
- **Lex Fridman Podcast** — Long-form conversations with AI leaders on safety philosophy and governance.
- **Into AI Safety (Kairos.fm)** — Accessible conversations about AI safety rules and regulations.
- **AI Safety Fundamentals: Alignment** — Educational podcast covering alignment fundamentals.
- **Doom Debates** — Rigorous debates on short timelines and high p(doom).

## Try another topic

Each call to `run_research` creates a new conversation, so the agent starts fresh with no context from the previous query. Edit `QUERY` and run the cell.

Example topics:
- "Find podcast episodes covering climate technology and clean energy innovations"
- "Research podcast interviews with startup founders about lessons learned from building companies"
- "Podcasts about the history and future of space exploration"

In [15]:
QUERY = "Find podcast episodes covering climate technology and clean energy innovations"

new_briefing = await run_research(QUERY)
display(Markdown(new_briefing))

............
[Tool call] search_episodes

[Tool call] search_episodes

[Tool call] search_podcasts
..........
[Tool call] get_podcast_details

[Tool call] get_podcast_details

[Tool call] get_podcast_details

[Tool call] get_podcast_details

[Tool call] get_podcast_details

[Tool call] get_podcast_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details

[Tool call] get_episode_details
.........
[Tool call] generate_research_briefing
..............................................................................................................................................................................................................................................................................................................................................................................................................................................................

I'll search for podcast episodes and shows covering climate technology and clean energy innovations using multiple queries to get diverse results. Let me also do a web search for additional context on the latest developments.Excellent! I've gathered a wealth of results. Now let me get details on some of the top podcast shows and specific episodes to enrich the data.I now have comprehensive data on podcasts, episodes, and web research. Let me compile everything into a research briefing.Here is your comprehensive research briefing on **Climate Technology and Clean Energy Innovations** — covering podcast episodes, shows, and the broader landscape.

---

## 🎧 Top Podcast Episodes on Climate Tech & Clean Energy

I searched Spotify and the web, identified the most relevant episodes and shows, and ranked them by relevance. Here are the highlights:

---

### ⭐ Top 10 Episodes

| # | Episode | Show | Duration | Date | Key Topic |
|---|---------|------|----------|------|-----------|
| 1 | [How Much Better Can Solar Get? — Martin Green](https://open.spotify.com/episode/3E7SXkyIuk8biB3fzLOJCi) | Cleaning Up | 52m | Aug 2026 | PERC/TOPCon solar breakthroughs, 90% of global panels |
| 2 | [From EVs to BESS: Why Capital Is Flowing to Batteries](https://open.spotify.com/episode/7cF4pDsmX0gMPDwnkJOote) | Redefining Energy | 21m | Aug 2026 | LFP dominance, 500 GW BESS by 2026, sodium batteries |
| 3 | [Demand for Power Gen & Grid Equipment Is Booming](https://open.spotify.com/episode/2nensCJ8gMsqqwmQCBaqek) | Energy Gang | 1h 9m | Sep 2026 | AI-data center demand, GE Vernova scaling to 30 GW/yr |
| 4 | [One Weird Trick to Get Unlimited Clean Energy](https://open.spotify.com/episode/4sH3rniFmHZiVwr4SLCuTj) | Unexplainable (Vox) | 25m | Jul 2025 | Enhanced geothermal systems |
| 5 | [What Role Can Green Hydrogen Play?](https://open.spotify.com/episode/097OM7OYZVwJ82nd5xPMjm) | All Things Renewable | 15m | Aug 2025 | IRENA experts on hydrogen's cost gap & scaling |
| 6 | [The Wind Turbine Built for Almost No Wind](https://open.spotify.com/episode/4KPtXlo3Fc9EMEmf9BtzLX) | The Clean Energy Show | 47m | Aug 2026 | China's low-wind turbine innovation |
| 7 | [How Tech Is Powering the Clean Energy Transition](https://open.spotify.com/episode/2kuBuSsAwoh45dCrLgpxBK) | Projectified | 29m | Nov 2024 | Grid connection, hydropower upgrades, net-zero tech |
| 8 | [Inside the Transition: Bottlenecks & Strategic Choices](https://open.spotify.com/episode/2FJF2uFktvy98j2SI8jwia) | IMD Sustainability | 43m | Dec 2025 | RWE's strategy across Shell/Siemens/RWE |
| 9 | [Unlocking the Future of Energy Tech — Schneider Electric](https://open.spotify.com/episode/5ZdFAWxKB9QBdZUtkl8z0C) | Supply Chain Now | 12m | Dec 2025 | AI-driven demand, grid modernization |
| 10 | [Decarbonising Buildings](https://open.spotify.com/episode/2GGAtH64qY4a6NTxbmvoxy) | Redefining Energy | 31m | Aug 2024 | Buildings = 37% of emissions; heating/cooling challenges |

---

### 📻 Top Podcast Shows to Follow

| Show | Episodes | Focus |
|------|----------|-------|
| [**Energy Gang**](https://open.spotify.com/show/0GT5BuD33AiPnqOzQE7YAE) (Wood Mackenzie) | 584 | Clean tech news, energy policy, hydrogen, nuclear, CCUS, EVs |
| [**Cleaning Up**](https://open.spotify.com/show/0msi0HzVNM05S6rMO5bbpn) | 303 | Weekly leadership conversations in clean energy & climate finance |
| [**Clean Power Hour**](https://open.spotify.com/show/4QpRZoiYIV72ZNz7hTYX0W) | 438 | Solar PV, battery storage, wind, wave — accelerating the transition |
| [**Redefining Energy**](https://open.spotify.com/show/4FDIRo16s1C9Fpc9v1HyGi) | 203 | Investment bankers on tech, finance, markets & regulations |
| [**Energy Evolution**](https://open.spotify.com/show/1hBfrTaP5TwpsIvx51fTN9) (S&P Global) | 364 | Decarbonization, emerging fuels, commodity markets |
| [**The Clean Energy Show**](https://open.spotify.com/show/7nIGJi8pZ9EmiJTPY4VOr5) | 296 | Climate change, EVs, solar, wind — accessible clean tech |

---

## 🔑 Key Themes Emerging Across Episodes

1. **AI as Both Driver and Solution** — The AI data-center boom is reshaping energy demand (GE Vernova investing $1.3B, scaling turbines to 30 GW/yr), while AI simultaneously enables grid optimization and climate tech R&D.

2. **Next-Generation Battery Storage** — LFP chemistry dominates at 95% BESS market share globally; sodium-ion is emerging; global BESS expected to hit 500 GW by end of 2026 and 1,000 GW by 2030.

3. **Solar's Continued Evolution** — Solar is the cheapest new electricity source globally, yet PERC and TOPCon innovations from Prof. Martin Green (found in 90%+ of panels) show the technology is still improving.

4. **Green Hydrogen's Narrowing Window** — Clear potential for heavy industry, aviation, and steel, but a "chicken-and-egg" investment dilemma and cost gap with fossil alternatives remain.

5. **Grid & Manufacturing Bottlenecks** — Supply chain constraints, permitting delays, and manufacturing capacity are critical barriers to scaling deployment.

6. **Expanding Renewable Geography** — Low-wind turbines (China) and enhanced geothermal systems are broadening where clean energy can be deployed.

7. **Hard-to-Abate Sectors** — Buildings account for 37% of emissions; lighting is a success story but heating/cooling remain complex.

---

## 🌍 Broader Industry Context (from Web Research)

- **Investment:** Global clean energy investment hit **$2.1 trillion in 2024** — double fossil fuel spending. US climate tech VC stayed flat at $14B. The market is projected to grow from $48B (2025) to **$313B by 2035**.
- **Solar Scale:** 597 GW installed in 2024 (+33%); 655 GW expected in 2025; solar PV to capture 80% of renewable capacity growth by 2030.
- **Wind:** 117 GW added globally in 2024.
- **AI-Energy Convergence:** Energy overtook transportation as the most funded US climate tech sector (39%, $5.4B), driven by AI electricity demand.
- **Emerging Tech:** Solid-state batteries, Tesla Megapack grid storage, Sweden's Hybit fossil-free steel project, low-carbon cement, chemical recycling, AI-powered waste sorting.

---

## 📋 Notable Experts Featured

| Expert | Role | Episode |
|--------|------|---------|
| **Prof. Martin Green** | UNSW; pioneer of PERC solar cells (in 90%+ of panels) | Cleaning Up |
| **Roger Martella** | Chief Sustainability Officer, GE Vernova | Energy Gang |
| **Kunal Chandra** | Chief Strategy Officer, RWE (ex-Shell, Siemens) | IMD Sustainability |
| **James Walker & Francisco Gaffaro** | IRENA team leads on green hydrogen | All Things Renewable |
| **Dylan Matthews** | Senior Correspondent, Vox Future Perfect | Unexplainable |

---

## ⚠️ Coverage Gaps Identified

- **No dedicated CCUS (carbon capture)** episodes despite being a major climate tech area
- **Limited nuclear energy innovation** coverage (SMRs, fusion) — mentioned in show descriptions but not in available episodes
- **No episodes on climate tech VC/investment trends** despite $2.1T market data
- **Minimal policy/regulatory coverage** (IRA, permitting reform, carbon pricing)
- **Geographic bias** toward Western and Chinese perspectives — limited Global South representation
- **No circular economy / green materials** episodes (low-carbon cement, chemical recycling)

---

Would you like me to dive deeper into any specific technology area (e.g., battery storage, green hydrogen, solar innovation) or search for additional episodes on a particular subtopic?

## Cleanup

Agents persist on Mistral's servers until deleted. You can delete the agent when you're done if you don't plan to use it. Any conversations associated with the agent are also cleaned up.

In [16]:
await client.beta.agents.delete_async(agent_id=agent.id)
print(f"Agent deleted: {agent.id}")

Agent deleted: ag_01a06bb33c63715b8dfec5b7bd6590bb


## Summary

This notebook demonstrated how to build a podcast research agent that searches Spotify, gathers web context, and generates structured briefings.

**What you built:**
- Six function tools (Spotify search + briefing generation) defined inline with tool schemas
- A Mistral agent that orchestrates podcast research across all tools and built-in web search
- A streaming pipeline that executes tool calls locally and renders the final briefing as markdown

**Mistral features used:**
- Agents API (beta)
- Conversations API (beta) with `FunctionCallEvent` / `FunctionResultEntry` for tool execution
- Built-in web search tool

**Other services:**
- [Spotify Web API](https://developer.spotify.com/documentation/web-api) — podcast catalog search via `spotipy`

Learn more about building agents in the [Agents documentation](https://docs.mistral.ai/studio/agents/introduction).